# PyTorch Autograd

Autograd is PyTorch's automatic differentiation engine.

It automatically calculates gradients (derivatives) of tensors with respect to other tensors.

Gradients are the foundation of:
- Backpropagation
- Neural network training
- Gradient Descent
- Optimization
- Deep Learning

The basic workflow is:

**Forward Pass → Calculate Loss → Backward Pass → Calculate Gradients → Update Parameters**

PyTorch mainly uses:
- `requires_grad=True`
- `.backward()`
- `.grad`
- Computational Graph

## 1. Understanding Derivatives

Before using PyTorch Autograd, let's understand a simple derivative.

Consider:

$$
y = x^2
$$

The derivative is:

$$
\frac{dy}{dx} = 2x
$$

So if:

$$
x = 3
$$

then:

$$
\frac{dy}{dx} = 2(3) = 6
$$

We can calculate this manually using Python.

In [59]:
def dy_dx(x):
    return 2 * x

dy_dx(3)

6

## 2. PyTorch Autograd

Instead of manually calculating derivatives, PyTorch can calculate them automatically.

To tell PyTorch that we want to calculate the gradient of a tensor, we use:

```python
requires_grad=True

In [60]:

### Code

import torch

x = torch.tensor(3.0, requires_grad=True)

y = x ** 2

x

tensor(3., requires_grad=True)

### Forward Pass

Here:

$$
y = x^2
$$

and:

$$
x = 3
$$

Therefore:

$$
y = 3^2 = 9
$$

PyTorch stores the operations needed to calculate the gradient.

In [61]:
y

tensor(9., grad_fn=<PowBackward0>)

## 3. Backward Pass

Now we ask PyTorch to calculate the derivative.

We use:

```python
y.backward()

In [62]:
x.grad

In [63]:

### Code

y.backward()

x.grad

tensor(6.)

## 4. Autograd and the Chain Rule

Consider:

$$
y = x^2
$$

and:

$$
z = \sin(y)
$$

Therefore:

$$
z = \sin(x^2)
$$

To calculate $\frac{dz}{dx}$, we use the chain rule:

$$
\frac{dz}{dx}
=
\frac{dz}{dy}
\frac{dy}{dx}
$$

We know:

$$
\frac{dz}{dy} = \cos(y)
$$

and:

$$
\frac{dy}{dx} = 2x
$$

Therefore:

$$
\frac{dz}{dx}
=
2x\cos(x^2)
$$

Autograd calculates this automatically.

In [64]:
import math

def dz_dx(x):
    return 2 * x * math.cos(x ** 2)

dz_dx(4)

-7.661275842587077

### Using PyTorch Autograd

We now calculate the same derivative using PyTorch.

The computational graph is:

$$
x \rightarrow y=x^2 \rightarrow z=\sin(y)
$$

Calling:

```python
z.backward()

In [65]:

### Code

x = torch.tensor(4.0, requires_grad=True)

y = x ** 2
z = torch.sin(y)

x

tensor(4., requires_grad=True)

In [66]:
y

tensor(16., grad_fn=<PowBackward0>)

In [67]:
z

tensor(-0.2879, grad_fn=<SinBackward0>)

### Backward Pass

Now calculate:

$$
\frac{dz}{dx}
$$

using:

```python
z.backward()

In [68]:

### Code

z.backward()

x.grad

tensor(-7.6613)

## 5. Leaf vs Non-Leaf Tensors

PyTorch distinguishes between **leaf tensors** and **non-leaf tensors**.

### Leaf Tensor

A tensor created directly by the user is generally a leaf tensor.

Example:

```python
x = torch.tensor(4.0, requires_grad=True)

In [69]:
y = x ** 2
z = torch.sin(y)

In [70]:
x.grad

tensor(-7.6613)

In [71]:
y.grad

/tmp/ipykernel_1494/486760323.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  y.grad


In [72]:

### Code

print("x.is_leaf:", x.is_leaf)
print("y.is_leaf:", y.is_leaf)
print("z.is_leaf:", z.is_leaf)

x.is_leaf: True
y.is_leaf: False
z.is_leaf: False


## 6. Getting the Gradient of a Non-Leaf Tensor

Sometimes we also want to see the gradient of an intermediate tensor.

For this, we can use:

```python
retain_grad()

In [73]:
y.retain_grad()

In [74]:

### Code

x = torch.tensor(4.0, requires_grad=True)

y = x ** 2
y.retain_grad()

z = torch.sin(y)

z.backward()

print("x.grad =", x.grad)
print("y.grad =", y.grad)

x.grad = tensor(-7.6613)
y.grad = tensor(-0.9577)


## 7. Manual Gradient Calculation vs Autograd

Let's compare manual differentiation with PyTorch Autograd.

Consider:

$$
z = \sin(x^2)
$$

The manual derivative is:

$$
\frac{dz}{dx}
=
2x\cos(x^2)
$$

Autograd should produce the same result.

This demonstrates that Autograd is performing the chain rule automatically.

In [75]:
x_value = 4.0

manual_gradient = dz_dx(x_value)

x = torch.tensor(x_value, requires_grad=True)

y = x ** 2
z = torch.sin(y)

z.backward()

autograd_gradient = x.grad.item()

print("Manual gradient:", manual_gradient)
print("Autograd gradient:", autograd_gradient)

Manual gradient: -7.661275842587077
Autograd gradient: -7.661275863647461


# 8. Autograd in Logistic Regression

Now let's see how Autograd is used in a machine learning model.

We have:

- Input: $x$
- Weight: $w$
- Bias: $b$

The linear equation is:

$$
z = wx + b
$$

The sigmoid function converts this into a probability:

$$
\hat{y} = \sigma(z)
$$

where:

$$
\sigma(z) = \frac{1}{1+e^{-z}}
$$

For binary classification, we use Binary Cross-Entropy (BCE) loss.

The complete flow is:

$$
x
\rightarrow
z=wx+b
\rightarrow
\hat{y}=\sigma(z)
\rightarrow
Loss
$$

During the backward pass, Autograd calculates:

$$
\frac{\partial L}{\partial w}
$$

and

$$
\frac{\partial L}{\partial b}
$$

In [76]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0)
b = torch.tensor(0.0)

## Binary Cross-Entropy Loss

For binary classification:

$$
L =
-\left[
y\log(\hat{y})
+
(1-y)\log(1-\hat{y})
\right]
$$

We use a small value `epsilon` to avoid taking:

$$
\log(0)
$$

In [77]:
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8

    prediction = torch.clamp(
        prediction,
        epsilon,
        1 - epsilon
    )

    return -(
        target * torch.log(prediction)
        + (1 - target) * torch.log(1 - prediction)
    )

## Forward Pass

First calculate the linear output:

$$
z = wx+b
$$

Then apply sigmoid:

$$
\hat{y} = \sigma(z)
$$

Finally calculate the BCE loss.

In [78]:
z = w * x + b

y_pred = torch.sigmoid(z)

loss = binary_cross_entropy_loss(y_pred, y)

print("z =", z)
print("Prediction =", y_pred)
print("Loss =", loss)

z = tensor(6.7000)
Prediction = tensor(0.9988)
Loss = tensor(6.7012)


## 9. Manual Gradient Calculation

For educational purposes, we can calculate the gradients manually.

The chain rule gives:

$$
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial \hat{y}}
\frac{\partial \hat{y}}{\partial z}
\frac{\partial z}{\partial w}
$$

Similarly:

$$
\frac{\partial L}{\partial b}
=
\frac{\partial L}{\partial \hat{y}}
\frac{\partial \hat{y}}{\partial z}
\frac{\partial z}{\partial b}
$$

The individual derivatives are:

$$
\frac{\partial L}{\partial \hat{y}}
=
\frac{\hat{y}-y}
{\hat{y}(1-\hat{y})}
$$

$$
\frac{\partial \hat{y}}{\partial z}
=
\hat{y}(1-\hat{y})
$$

$$
\frac{\partial z}{\partial w}=x
$$

$$
\frac{\partial z}{\partial b}=1
$$

In [79]:
dloss_dy_pred = (
    (y_pred - y)
    / (y_pred * (1 - y_pred))
)

dy_pred_dz = y_pred * (1 - y_pred)

dz_dw = x
dz_db = 1

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

print("Manual Gradient of loss w.r.t weight:", dL_dw)
print("Manual Gradient of loss w.r.t bias:", dL_db)

Manual Gradient of loss w.r.t weight: tensor(6.6918)
Manual Gradient of loss w.r.t bias: tensor(0.9988)


## 10. Letting Autograd Calculate the Gradients

Instead of calculating the derivatives manually, we can tell PyTorch to track `w` and `b`.

We set:

```python
requires_grad=True

In [80]:

### Code

x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [81]:
z = w * x + b
y_pred = torch.sigmoid(z)

loss = binary_cross_entropy_loss(y_pred, y)

print("z:", z)
print("prediction:", y_pred)
print("loss:", loss)

z: tensor(6.7000, grad_fn=<AddBackward0>)
prediction: tensor(0.9988, grad_fn=<SigmoidBackward0>)
loss: tensor(6.7012, grad_fn=<NegBackward0>)


In [82]:
loss.backward()

print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


The gradients obtained through Autograd should match the manually calculated gradients.

This is the basic mechanism behind backpropagation in neural networks.

# 11. Gradient Accumulation

An important property of PyTorch is that gradients are **accumulated**.

For example, if we call:

```python
backward()

In [83]:

### Code

x = torch.tensor(2.0, requires_grad=True)

y = x ** 2

y.backward()

print("First gradient:", x.grad)

First gradient: tensor(4.)


The gradient is:

$$
\frac{dy}{dx}=2x
$$

At $x=2$:

$$
\frac{dy}{dx}=4
$$

In [84]:
x.grad.zero_()

print("Gradient after clearing:", x.grad)

Gradient after clearing: tensor(0.)


## 12. Clearing Gradients

In neural network training, the usual pattern is:

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()


---

# 12. Gradients with Multiple Elements

## Cell 25

### Markdown

```markdown
# 13. Autograd with Tensors

Autograd also works with tensors containing multiple values.

Consider:

$$
x = [1,2,3]
$$

and:

$$
y = mean(x^2)
$$

Since:

$$
y = \frac{1^2+2^2+3^2}{3}
$$

Autograd can calculate the gradient of `y` with respect to every element of `x`.

In [85]:
x = torch.tensor(
    [1.0, 2.0, 3.0],
    requires_grad=True
)

y = (x ** 2).mean()

print("x:", x)
print("y:", y)

x: tensor([1., 2., 3.], requires_grad=True)
y: tensor(4.6667, grad_fn=<MeanBackward0>)


In [86]:
y.backward()

x.grad

tensor([0.6667, 1.3333, 2.0000])

# 14. Controlling Gradient Tracking

`requires_grad` tells PyTorch whether a tensor should participate in gradient tracking.

Example:

```python
x = torch.tensor(2.0, requires_grad=True)

In [87]:

### Code

x = torch.tensor(2.0, requires_grad=True)

print(x.requires_grad)

True


# 15. Disabling Gradient Tracking

There are three important ways to stop or control gradient tracking.

### 1. `requires_grad_(False)`

Changes the `requires_grad` property of the tensor.

### 2. `detach()`

Creates a tensor that shares the same data but is disconnected from the computational graph.

### 3. `torch.no_grad()`

Temporarily disables gradient tracking inside a block of code.

These are especially useful during model evaluation/inference.

In [88]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2

y.backward()

print("Before disabling:", x.grad)

x.requires_grad_(False)

print("requires_grad:", x.requires_grad)

Before disabling: tensor(4.)
requires_grad: False


## 16. Using `detach()`

`detach()` creates a new tensor that is disconnected from the current computational graph.

Example:

```python
z = x.detach()

In [89]:

### Code

x = torch.tensor(2.0, requires_grad=True)

y = x ** 2

z = x.detach()

print("x:", x)
print("x.requires_grad:", x.requires_grad)

print("z:", z)
print("z.requires_grad:", z.requires_grad)

x: tensor(2., requires_grad=True)
x.requires_grad: True
z: tensor(2.)
z.requires_grad: False


In [90]:
y1 = z ** 2

print("y:", y)
print("y1:", y1)

y: tensor(4., grad_fn=<PowBackward0>)
y1: tensor(4.)


In [91]:
y1 = z ** 2


---

# 18. `torch.no_grad()`

## Cell 32

### Markdown

```markdown
# 17. `torch.no_grad()`

During inference/testing, we usually don't need gradients.

For example:

```python
with torch.no_grad():
    output = model(input)

In [92]:

### Code


x = torch.tensor(2.0, requires_grad=True)

with torch.no_grad():
    y = x ** 2

print(y)
print(y.requires_grad)

tensor(4.)
False


# 18. `torch.inference_mode()`

PyTorch also provides:

```python
torch.inference_mode()

In [93]:

### Code


x = torch.tensor(2.0, requires_grad=True)

with torch.inference_mode():
    y = x ** 2

print(y)
print(y.requires_grad)

tensor(4.)
False


# 19. Why Does `backward()` Usually Need a Scalar?

For a scalar:

```python
y = x ** 2
y.backward()

In [94]:

### Code

x = torch.tensor(
    [1.0, 2.0, 3.0],
    requires_grad=True
)

y = x ** 2

gradient = torch.tensor([1.0, 1.0, 1.0])

y.backward(gradient)

x.grad

tensor([2., 4., 6.])

# 20. Computational Graph

Autograd works by constructing a computational graph.

For:

```python
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
z = 3 * y


---

# 22. `grad_fn`

## Cell 36

### Markdown

```markdown
# 21. Understanding `grad_fn`

Every tensor produced through a tracked operation can contain information about the operation that created it.

We can inspect this using:

```python
tensor.grad_fn

In [95]:

### Code

x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
z = 3 * y

print("x.grad_fn:", x.grad_fn)
print("y.grad_fn:", y.grad_fn)
print("z.grad_fn:", z.grad_fn)

x.grad_fn: None
y.grad_fn: <PowBackward0 object at 0x7e9c56746f80>
z.grad_fn: <MulBackward0 object at 0x7e9c56746f80>


# 22. Higher-Order Derivatives

Autograd can calculate not only first-order derivatives but also higher-order derivatives.

Consider:

$$
y=x^3
$$

First derivative:

$$
\frac{dy}{dx}=3x^2
$$

Second derivative:

$$
\frac{d^2y}{dx^2}=6x
$$

To calculate higher-order derivatives, we need:

```python
create_graph=True

In [96]:

### Code

x = torch.tensor(2.0, requires_grad=True)

y = x ** 3

first_grad = torch.autograd.grad(
    y,
    x,
    create_graph=True
)[0]

print("First derivative:", first_grad)

First derivative: tensor(12., grad_fn=<MulBackward0>)


In [97]:
second_grad = torch.autograd.grad(
    first_grad,
    x
)[0]

print("Second derivative:", second_grad)

Second derivative: tensor(12.)


# 23. `torch.autograd.grad()`

Another way of calculating gradients is:

```python
torch.autograd.grad()

In [98]:

### Code

x = torch.tensor(3.0, requires_grad=True)

y = x ** 2

gradient = torch.autograd.grad(y, x)

print(gradient)

(tensor(6.),)


# 24. Autograd + Gradient Descent

Autograd calculates gradients, but it does not automatically update the model parameters.

Gradient Descent uses the gradient to update parameters.

The basic update rule is:

$$
w_{new}=w_{old}-\eta\frac{\partial L}{\partial w}
$$

where:

- $w$ = parameter
- $L$ = loss
- $\eta$ = learning rate

The complete training process is:

1. Forward pass
2. Calculate loss
3. Backward pass
4. Calculate gradients
5. Update parameters
6. Clear gradients
7. Repeat

This is the core training loop of neural networks.

In [99]:
x = torch.tensor(2.0)

w = torch.tensor(
    1.0,
    requires_grad=True
)

learning_rate = 0.1

for epoch in range(5):

    # Forward pass
    y = w * x

    # Loss
    loss = (y - 4) ** 2

    # Backward pass
    loss.backward()

    # Update parameter
    with torch.no_grad():
        w -= learning_rate * w.grad

    # Clear gradient
    w.grad.zero_()

    print(
        f"Epoch {epoch + 1}: "
        f"w = {w.item():.4f}, "
        f"loss = {loss.item():.4f}"
    )

Epoch 1: w = 1.8000, loss = 4.0000
Epoch 2: w = 1.9600, loss = 0.1600
Epoch 3: w = 1.9920, loss = 0.0064
Epoch 4: w = 1.9984, loss = 0.0003
Epoch 5: w = 1.9997, loss = 0.0000


# 25. Autograd with PyTorch Optimizer

In real neural networks, we normally don't manually write:

```python
w -= learning_rate * w.grad

In [100]:
import torch

x = torch.tensor(2.0)

w = torch.tensor(
    1.0,
    requires_grad=True
)

optimizer = torch.optim.SGD(
    [w],
    lr=0.1
)

In [101]:
for epoch in range(5):

    # Forward pass
    y = w * x

    # Loss
    loss = (y - 4) ** 2

    # Clear previous gradients
    optimizer.zero_grad()

    # Backward pass
    loss.backward()

    # Update weight
    optimizer.step()

    print(
        f"Epoch {epoch + 1}: "
        f"w = {w.item():.4f}, "
        f"loss = {loss.item():.4f}"
    )

Epoch 1: w = 1.8000, loss = 4.0000
Epoch 2: w = 1.9600, loss = 0.1600
Epoch 3: w = 1.9920, loss = 0.0064
Epoch 4: w = 1.9984, loss = 0.0003
Epoch 5: w = 1.9997, loss = 0.0000


# 26. Autograd — Final Summary

## What is Autograd?

Autograd is PyTorch's automatic differentiation system.

It automatically calculates gradients required for backpropagation.

---

## Important Concepts

### `requires_grad=True`

Tells PyTorch to track operations involving the tensor.

```python
x = torch.tensor(2.0, requires_grad=True)